In [240]:
library(torch)
library(luz)
#library(lecospectR)

In [276]:
net <- nn_module(
    "jigsaw",
     initialize = function(num_components = NUM_VARS, num_levels = 8) {

      self$A <- nn_conv1d(
        in_channels = num_components, 
        out_channels = 1, 
        kernel_size = 1)

      # define the B block;
      self$B3 <- nn_sequential(
        nn_conv1d(num_components,1,1), 
        nn_conv1d(1,1,3))
      self$B5 <- nn_sequential(
        nn_conv1d(num_components,num_components,1), 
        nn_conv1d(num_components,num_components,5))
      self$B7 <- nn_sequential(
        nn_conv1d(num_components,num_components,1), 
        nn_conv1d(num_components,num_components,7))
      self$B9 <- nn_sequential(
        nn_conv1d(num_components,num_components,1), 
        nn_conv1d(num_components,num_components,9))

    self$B <- nn_sequential(
      nn_adaptive_avg_pool1d(
        num_components
      ), 
      nn_flatten(), 
      nn_linear(
        in_features = num_components,
        out_features = num_components
      ), 
      nn_dropout(p = 0.4)
      )



      #C block
      self$C <- nn_sequential(
        nn_flatten(start_dim = 2,end_dim = -1),
        nn_linear(
          in_features = 2, 
          out_features = num_components
        ), 
        nn_dropout(p = 0.4),
        nn_linear(
          in_features = num_components, 
          out_features = num_components
        ),
        nn_dropout(p = 0.4)
      )

      self$D <- nn_sequential(
        nn_linear(
          in_features = num_components,
          out_features = num_components
        ),
        nn_dropout(p = 0.4),
        nn_linear(
          in_features = num_components,
          out_features = num_components
        ),
        nn_softmax(2)
      )
    
  },
  forward = function(x) {
    x_A <- self$A(x)
    x_b3 <- self$B3(x_A)
    x_b5 <- self$B5(x_A)
    x_b7 <- self$B7(x_A)
    x_b9 <- self$B9(x_A)

    x_B_concat <- torch_cat(tensors = list(
      x_b3, x_b5, x_b7, x_b9
    ))

    x_B <- self$B(x_B_concat)

    x_C <- self$C(x_A)



    x_D <- self$D(
      torch_cat(tensors = list(
        x_B, x_C
      ))
    )

    return(x_D)
  }

  
)

In [243]:
add_channel_dim <- function(img) img$unsqueeze(1)

In [244]:
create_variable_names <- function(start, stop){
    names <- c()
    for(i in start:stop){
        names <- append(
            names, 
            paste0("X", i))
    }

    return(names)
}

In [245]:
independent_variables <- create_variable_names(400L, 999L)

dependent_variable <- "FncGrp1" 
NUM_VARS <- length(independent_variables)
print(independent_variables)

  [1] "X400" "X401" "X402" "X403" "X404" "X405" "X406" "X407" "X408" "X409"
 [11] "X410" "X411" "X412" "X413" "X414" "X415" "X416" "X417" "X418" "X419"
 [21] "X420" "X421" "X422" "X423" "X424" "X425" "X426" "X427" "X428" "X429"
 [31] "X430" "X431" "X432" "X433" "X434" "X435" "X436" "X437" "X438" "X439"
 [41] "X440" "X441" "X442" "X443" "X444" "X445" "X446" "X447" "X448" "X449"
 [51] "X450" "X451" "X452" "X453" "X454" "X455" "X456" "X457" "X458" "X459"
 [61] "X460" "X461" "X462" "X463" "X464" "X465" "X466" "X467" "X468" "X469"
 [71] "X470" "X471" "X472" "X473" "X474" "X475" "X476" "X477" "X478" "X479"
 [81] "X480" "X481" "X482" "X483" "X484" "X485" "X486" "X487" "X488" "X489"
 [91] "X490" "X491" "X492" "X493" "X494" "X495" "X496" "X497" "X498" "X499"
[101] "X500" "X501" "X502" "X503" "X504" "X505" "X506" "X507" "X508" "X509"
[111] "X510" "X511" "X512" "X513" "X514" "X515" "X516" "X517" "X518" "X519"
[121] "X520" "X521" "X522" "X523" "X524" "X525" "X526" "X527" "X528" "X529"
[131] "X530"

In [254]:
# define the dataset
if(is.null(data_df)){
    data_df <- read.csv(file = "Data/Ground_Validation/PFT_image_spectra/PFT_Image_SpectralLib_Clean.csv")
}


In [247]:

lecospec_dataset <- torch::dataset(
    name = "lecospec_image",
    initialize = function(df) {
        self$x <- as.matrix(df[, independent_variables])
        self$y <- torch_tensor(
            as.matrix(
                as.numeric(
                    #mltools::one_hot(
                    as.factor(
                        df[, dependent_variable]
                        )
                    #)
                )
                )
            )
    },

    .getitem = function(i) {
        list(
            x = torch_flatten(self$x[i,]),
            y = torch_flatten(self$y[i])
        )
    },

    .length = function() {
        self$y$size()[[1]]
    }
)

In [248]:
head(data_df)

,X,UID,ScanNum,sample_name,PFT,FncGrp1,Site,X398,X399,X400,⋯,X990,X991,X992,X993,X994,X995,X996,X997,X998,X999
,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,BisonGulchPFTsBetula1,1,spec_1,Betula,ShrubDecid,BisonGulch,0.05814769,0.05926529,0.06028869,⋯,0.6815182,0.6811660,0.6890470,0.7040298,0.7249807,0.7507566,0.7801884,0.8121027,0.8453261,0.8786852
2,2,BisonGulchPFTsBetula1,1,spec_2,Betula,ShrubDecid,BisonGulch,0.04456014,0.04778814,0.05079318,⋯,0.6706666,0.6683159,0.6786394,0.7000307,0.7308801,0.7695067,0.8140391,0.8625739,0.9132079,0.9640378
3,3,BisonGulchPFTsBetula1,1,spec_3,Betula,ShrubDecid,BisonGulch,0.03929324,0.04265593,0.04557066,⋯,0.5152525,0.5091915,0.5178217,0.5395294,0.5726982,0.6156166,0.6663192,0.7227978,0.7830447,0.8450520
4,4,BisonGulchPFTsBetula1,1,spec_4,Betula,ShrubDecid,BisonGulch,0.13230228,0.11122692,0.09129034,⋯,0.5120581,0.5113880,0.5348292,0.5745538,0.6227243,0.6723311,0.7185860,0.7570701,0.7833644,0.7930498
5,5,BisonGulchPFTsBetula1,1,spec_5,Betula,ShrubDecid,BisonGulch,0.05211388,0.05565497,0.05878525,⋯,0.6863419,0.6680365,0.6509006,0.6344450,0.6181806,0.6017555,0.5851848,0.5685449,0.5519121,0.5353626
6,6,BisonGulchPFTsBetula1,1,spec_6,Betula,ShrubDecid,BisonGulch,0.06955397,0.06788242,0.06631141,⋯,0.7354495,0.7371508,0.7445194,0.7567953,0.7732173,0.7930235,0.8154512,0.8397375,0.8651196,0.8908347


In [249]:
train_indices <- sample(1:nrow(data_df), 250)
train_ds <- lecospec_dataset(data_df[train_indices, ])
valid_ds <- lecospec_dataset(data_df[setdiff(1:nrow(data_df), train_indices), ])

length(train_ds)
length(valid_ds)

[1] 250

[1] 16951

In [278]:
# create a dataloader
test_dl <- torch::dataloader(
    valid_ds,
    batch_size = 64,
    shuffle = TRUE
)

train_dl <- torch::dataloader(
    train_ds,
    batch_size = 64,
    shuffle = TRUE
)

In [279]:
batch <- dataloader_make_iter(train_dl) %>% dataloader_next()
print(batch$x)
print(batch$y)

torch_tensor
Columns 1 to 10 0.0296  0.0298  0.0299  0.0301  0.0303  0.0304  0.0306  0.0307  0.0309  0.0311
 0.0367  0.0368  0.0369  0.0370  0.0371  0.0372  0.0373  0.0374  0.0375  0.0376
 0.0431  0.0433  0.0435  0.0437  0.0439  0.0441  0.0443  0.0445  0.0447  0.0449
 0.0357  0.0359  0.0362  0.0365  0.0368  0.0371  0.0373  0.0376  0.0379  0.0382
 0.0370  0.0369  0.0368  0.0367  0.0366  0.0364  0.0363  0.0362  0.0361  0.0360
 0.0221  0.0221  0.0221  0.0221  0.0221  0.0220  0.0220  0.0220  0.0220  0.0220
 0.0802  0.0799  0.0796  0.0793  0.0790  0.0787  0.0784  0.0781  0.0778  0.0775
 0.0382  0.0384  0.0386  0.0388  0.0390  0.0392  0.0395  0.0397  0.0399  0.0401
 0.0312  0.0315  0.0318  0.0321  0.0325  0.0328  0.0331  0.0334  0.0337  0.0340
 0.0218  0.0219  0.0220  0.0221  0.0221  0.0222  0.0223  0.0224  0.0225  0.0226
 0.0437  0.0447  0.0456  0.0466  0.0476  0.0485  0.0495  0.0505  0.0514  0.0524
 0.0404  0.0404  0.0404  0.0404  0.0404  0.0403  0.0403  0.0403  0.0403  0.0402
 0.0500  0.0

In [272]:
print(batch$x$shape)

[1] 250 600


In [280]:
model <- net()
model(batch$x)

ERROR: Error in (function (input, weight, bias, stride, padding, dilation, groups) : Given groups=1, weight of size [1, 600, 1], expected input[1, 64, 600] to have 600 channels, but got 64 channels instead
Exception raised from check_shape_forward at ../aten/src/ATen/native/Convolution.cpp:672 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x6b (0x7ed2900e505b in /usr/lib/R/site-library/torch/lib/libc10.so)
frame #1: c10::detail::torchCheckFail(char const*, char const*, unsigned int, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> > const&) + 0xbf (0x7ed2900dff6f in /usr/lib/R/site-library/torch/lib/libc10.so)
frame #2: <unknown function> + 0x16f4551 (0x7ed26def4551 in /usr/lib/R/site-library/torch/lib/libtorch_cpu.so)
frame #3: at::native::_convolution(at::Tensor const&, at::Tensor const&, c10::optional<at::Tensor> const&, c10::ArrayRef<long>, c1

In [ ]:
net_fitted <- net %>%
  setup(
    loss = nn_cross_entropy_loss(),
    optimizer = optim_adam
  ) %>%
  fit(train_dl, epochs = 1, valid_data = test_dl)

In [ ]:
net_fitted

A `luz_module_fitted`
── Time ────────────────────────────────────────────────────────────────────────
• Total time: 31.6s
• Avg time per training epoch: 999ms

── Results ─────────────────────────────────────────────────────────────────────
Metrics observed in the last epoch.

ℹ Training:
loss: 0

── Model ───────────────────────────────────────────────────────────────────────
An `nn_module` containing 257 parameters.

── Modules ─────────────────────────────────────────────────────────────────────
• net: <nn_sequential> #257 parameters

In [ ]:
predict(net_fitted,newdata = batch$x)

torch_tensor
 1
[ CPUFloatType{1,1} ]

## Next

Need to define the correct nueral net architecture, and then the training loop should work as intended.  This is likely some kind of U-Net from a paper.  

Just a matter of finding the correct one and plugging the layers into the model above.  

Hopefully that works directly